In [37]:

from pypdf import PdfReader
import os,json
import getpass
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
import fitz
from PIL import Image
import io
import base64

In [34]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)


In [56]:
evaluation_prompt = """
You are an expert in **AI Design Pattern Evaluation**.
Your task is to compare both lists and evaluate how accurately the generated patterns match the correct ones in **meaning and structure**.

Compare two lists of AI design patterns:

1. CorrectPatterns – verified ground truth.
2. GeneratedPatterns – AI-generated list.

Match patterns **semantically** by Name, Problem, Context, and Solution (not exact text).  
Minor wording changes are fine if the meaning is the same.

Return only this JSON:

  "precision": float,
  "recall": float,
  "f1": float,
  "correct": ["patterns correctly matched between both lists"],
  "missing": ["patterns in CorrectPatterns but not in GeneratedPatterns"],
  "extra": ["patterns in GeneratedPatterns but not in CorrectPatterns"]


Rules:
- precision = correct_matches / total_generated  
- recall = correct_matches / total_correct  
- f1 = 2 * (precision * recall) / (precision + recall)  
- Round numbers to 3 decimals.  
- Output valid JSON only.

Data:
CorrectPatterns:
{correct_json}

GeneratedPatterns:
{generated_json}
"""


In [36]:
correct_patterns = """Passive goal creator - Analyse users’ articulated prompts through the dialogue interface to preserve interactivity, goal-seeking and efficiency.
Proactive goal creator	- Anticipate users’ goals by understanding human interactions and capturing the context via relevant tools, to enhance interactivity, goal-seeking and accessibility.
Prompt/response optimiser - Optimise the prompts/responses according to the desired input or output content and format to provide standardisation, goal alignment, interoperability and adaptability.
Retrieval augmented generation- Enhance the knowledge updatability of the agents while maintaining data privacy of on-premise foundation model-based agents/systems implementations.
One-shot model querying - Access the foundation model in a single instance to generate all necessary steps for the plan for cost efficiency and simplicity.
Incremental model querying - Access the foundation model at each step of the plan generation process to provide supplementary context, improve reasoning certainty and explainability.
Single-path plan generator- Orchestrate the generation of intermediate steps leading to the achievement of the user’s goal to improve reasoning certainty, coherence and efficiency.
Multi-path plan generator - Allow multiple choice creation at each intermediate step leading to achieving users’ goals to enhance reasoning certainty, coherence, alignment to human preference and inclusiveness.
Self-reflection	- Enable the agent to generate feedback on the plan and reasoning process and provide refinement guidance from themselves to improve reasoning certainty, explainability, continuous improvement and efficiency.
Cross-reflection - Use different agents or foundation models to provide feedback and refine the generated plan and reasoning process for better reasoning certainty, explainability, inclusiveness and scalability.
Human reflection - Collect feedback from humans to refine the plan and reasoning process, to effectively align with human preference, improving contestability and effectiveness.
Voting-based cooperation - Enable free opinions expression across agents and reach consensus by submitting their votes to preserve fairness, accountability and collective intelligence.
Role-based cooperation -Assign assorted roles, and finalise decisions in accordance with the roles of agents for to facilitate division of labour, fault tolerance, scalability and accountability.
Debate-based cooperation - Provide and receive feedback across multiple agents adjusts the thoughts and behaviours during the debate with other agents until a consensus is reached to improve adaptability, explainability and critical thinking.
Multimodal guardrails	- Control the inputs and outputs of foundation models to meet specific requirements such as user requirements, ethical standards, and laws to enhance robustness, safety, standard alignment, and adaptability.
Tool/agent registry - Maintain a unified and convenient source to select diverse agents and tools to improve discoverability, efficiency, tool appropriateness and scalability.
Agent adapter	- Provide interface to connect the agent and external tools for task completion, ensuring interoperability and adaptability, and reduce development cost.
Agent evaluator - Perform testing to assess the agent regarding diverse requirements and metrics, ensuring the functional suitability, adaptability with improved flexibility.
"""

In [48]:
correct_patterns_2 = """Different Workloads in Different Computing Environments
Problem: It is necessary to separate and quickly changet he ML data workload and stabilize the training workload to maximize efficiency
Solution: Physically isolate different workloads to separate machines.Then optimize the machine configurations and thenetwork usage.
Distinguish BusinessLogic from MLModels7,11
Problem: The overall business logic should be isolated as much as possible from the ML models so that they can be changed/overridden as necessary without impacting the rest of the business logic.
Solution: Separate the business logic and the inference engine, loosely coupling the business logic and ML-specific dataflows.
ML Gateway Routing Architecture
When a client uses multiple services, it can be difficult to set up and manage individual endpoints for each service.
Install a gateway before a set of applications, services, ordeployments. Use application layer routing requests to theappropriate instance.
Microservice Architecture for ML
ML applications may be confined to some “known” ML frameworks, missing opportunities for more appropriate frameworks.
Define consistent input and output data. Provide well-defined services to use for ML frameworks.
Lambda Architecturefor ML
Real-time data processing requires scalability, fault tolerance, predictability, and other qualities.It must be extensible.
The batch layer keeps producing views at every set batch interval, while the speed layer creates the relevant real-time/speed views. The serving layer orchestrates the query by querying both the batch and speed layer, and then merging them.
Kappa Architecture for ML
It is necessary to deal with a huge amount of data with less code resources. 
Support both real-time data processing and continuous processing with a single stream processing engine.
Data Lake for ML
We cannot foresee the kind of analyses that will be performed on the data and which frameworks will be used to perform such analyses.
Store data, which range from structured to unstructured, as“raw” as possible into a data storage.
Separation of Concerns and Modularization of ML Components
ML applications must accommodate regular andfrequent changes to their ML components.Decouple at different levels of complexity from the simplest tothe most complex.
Encapsulate ML Models within Rule-base Safeguards
Problem: ML models are known to be unstable and vulnerable to adversarial attacks, noise in data, and data drift overtime.
Solution: Encapsulate functionality provided by ML models and appropriately deal with the inherent uncertainty of their outcomes in the containing system using deterministic and verifiable rules.
Discard PoC Code
Problem: The code created for PoC often includes code that sacrifices maintainability for efficient implementation of trial and error, and code that is ultimately no longer needed.
Solution: Discard the code created for the PoC and rebuild maintainable code based on the findings from the PoC.
Parameter-Server Abstraction
Problem: For distributed learning, widely accepted abstractions are lacking.
Solution: Distribute both data and workloads over worker nodes, while the server nodes maintain globally shared parameters, which are represented as vectors and matrices.
Data Flows Up, Model Flows Down
Problem: Standard ML approaches require centralizing the training data on one machine or in a datacenter.
Solution: Enable mobile devices to collaboratively learn a shared prediction model in the cloud while keeping all of the training data on the device as federated learning.
Secure Aggregation
Problem: The system needs to communicate and aggregate model updates in a secure, efficient, scalable, and fault-tolerant way.
Solution: Encrypt data from each mobile device in collaborative learning and calculate totals and averages without individual examination.
Deployable Canary Model
Problem: A surrogate ML that approximates the behavior of the best ML model must be built to provide explainability.
Solution: Run the explainable inference pipeline in parallel with the primary inference pipeline to monitor prediction differences.
ML Versioning
Problem: ML models and their different versions may change the behavior of the overall ML applications.
Solution: Record the ML model structure, training data set, training system and analytical code to ensure a reproducible training process and an inference process.
"""

In [60]:
correct_patterns_3 = """
| Pattern Number | Pattern Name | Problem | Solution | Usage Scenarios | Code Example |
| -------------: | :----------- | :------ | :------- | :-------------- | :----------- |
| 1 | Logits Masking | Need to ensure generated text conforms to specific style rules for brand, accuracy, or compliance reasons. | Intercept the generation at the sampling stage to zero out probabilities of continuations that don't meet the rules | Use words associated with specific brand; avoid repeating factual information; make content compliant with style book | [examples/01_logits_masking](examples/01_logits_masking)|
| 2 | Grammar | Need text to conform to a specific format or data schema for downstream processing. | Specify rules as a formal grammar (e.g., BNF) or schema that the model framework applies to constrain token generation. | Generating valid SQL timestamps; extracting structured data in a specific format; ensuring output conforms to JSON schema. | [examples/02_grammar](examples/02_grammar) |
| 3 | Style Transfer | Need to convert content into a form that mimics specific tone and style that is difficult to express through rules, but can be shown through example conversions. | Use few-shot learning or model fine-tuning to teach the model how to convert content to the desired style. | Rewriting generic content to match brand guidelines; converting academic papers to blog posts; transforming image and text content for different social media platforms or audiences. | [examples/03_style_transfer](examples/03_style_transfer) |
| 4 | Reverse Neutralization | Need to generate content in a specific style that can be shown through example content. | Use an LLM to generate content in an intermediate neutral form, and a fine-tuned LLM to convert that neutral form into the desired style. | Generating letters in region-specific legalese; generating emails in personal style. | [examples/04_reverse_neutralization](examples/04_reverse_neutralization) |
| 5 | Content Optimization | Need to determine optimal style for content without knowing which factors matter. | Generate pairs of content, compare them using an evaluator, create a preference dataset, and perform preference tuning. | Optimizing ad copy, marketing content, or educational materials where effective style factors are unknown. | [examples/05_content_optimization](examples/05_content_optimization) |

</details>

<details>
<summary>Chapters 3 and 4: Adding Knowledge (Patterns 6-12) </summary>
  
| Pattern Number | Pattern Name | Problem | Solution | Usage Scenarios | Code Example |
| -------------: | :----------- | :------ | :------- | :-------------- | :----------- |
| 6 | Basic RAG | Knowledge cutoff, confidential data, and hallucinations pose problems for zero-shot generation by LLMs. | Ground the response generated by the LLM by adding relevant information from a knowledge base into the prompt context. | The applications of RAG are constantly expanding as the technology evolves. | [examples/06_basic_rag](examples/06_basic_rag) |
| 7 | Semantic Indexing | Traditional keyword indexing/lookup approaches fail when documents get more complex, contain different media types like images or tables, or bridge multiple domains. | Use embeddings to capture the meaning of texts, images, and other media types. Find relevant chunks by comparing the embedding of the chunk to that of the query. | | [examples/07_semantic_indexing](examples/07_semantic_indexing) |
| 8 | Indexing at Scale | Dealing with outdated or contradictory information in your knowledge base. | Using metadata, query filtering, and result reranking. | | [examples/08_indexing_at_scale](examples/08_indexing_at_scale) |
| 9 | Index-aware Retrieval | Comparing questions to chunks is problematic because the question itself will not appear in the knowledge base, may use synonyms or jargon, or may require holistic interpretation. | Hypothetical answers, query expansion, hybrid search, GraphRAG | | [examples/09_index_aware_retrieval](examples/09_index_aware_retrieval) |
| 10 | Node Postprocessing | Irrelevant content, ambiguous entities, generic answers. | Reranking offer the ability to bring in a lot of other neat ideas: hybrid search, query expansion, filtering, contextual compression, disambiguation, personalization | | [examples/10_node_postprocessing](examples/10_node_postprocessing) |
| 11 | Trustworthy Generation | How to retain users’ trust given that there is no way to completely avoid errors. | Out-of-domain detection, citations, guardrails, human feedback, corrective RAG, UX design can all help. | | [examples/11_trustworthy_generation](examples/11_trustworthy_generation) |
| 12 | Deep Search | RAG systems are less effective for complex information retrieval tasks because of context window constraints, query ambiguity, information verification, shallow reasoning, and multi-hop query challenges. | Iterative process of searching, reading, and reasoning to provide comprehensive answers to complex queries. | | [examples/12_deep_search](examples/12_deep_search) |

</details>

<details>
<summary>Chapter 5: Extending Model Capabilities (Patterns 13-16) </summary>
  
| Pattern Number | Pattern Name | Problem | Solution | Usage Scenarios | Code Example |
| -------------: | :----------- | :------ | :------- | :-------------- | :----------- |
| 13 | Chain of Thought (CoT) | Foundational models often struggle with multi-step reasoning tasks, leading to incorrect or fabricated answers. | CoT prompts the model to break down complex problems into intermediate reasoning steps before providing the final answer. | Complex mathematical problems, logical deductions, and sequential reasoning tasks where step-by-step thinking is required. | [examples/13_chain_of_thought](examples/13_chain_of_thought) |
| 14 | Tree of Thoughts (ToT) | Many strategic or logical tasks cannot be solved by a single linear reasoning path, requiring exploration of multiple alternatives. | ToT treats problem-solving as a tree search, generating multiple reasoning paths, evaluating them, and backtracking as needed | Complex tasks involving strategic thinking, planning, or creative writing that require exploring multiple solution paths. | [examples/14_tree_of_thoughts](examples/14_tree_of_thoughts) |
| 15 | Adapter Tuning | Fully fine-tuning large foundational models for specialized tasks is computationally expensive and requires significant data.nt. | Adapter Tuning trains small add-on neural network layers, leaving the original model weights frozen, making it efficient for specialized adaptation. | Adapting models for specific tasks like classification, summarization, or specialized chatbots with a small (100-10k) dataset of examples. | [examples/15_adapter_tuning](examples/15_adapter_tuning) |
| 16 | Evol-Instruct | Creating high-quality datasets for instruction tuning models on new and complex enterprise tasks is difficult and time-consuming. | Evol-Instruct efficiently generates instruction-tuning datasets by evolving instructions through multiple iterations of LLM-generated tasks and answers. | Teaching models new, domain-specific tasks that are not covered by their pre-training data, particularly in enterprise settings. | [examples/16_evol_instruct](examples/16_evol_instruct) |

</details>

<details>
<summary>Chapter 6: Improving Reliability (Patterns 17-20) </summary>

| Pattern Number | Pattern Name | Problem | Solution | Usage Scenarios | Code Example |
| -------------: | :----------- | :------ | :------- | :-------------- | :----------- |  
| 17 | LLM-as-Judge | Evaluation of GenAI capabilities is hard because the tasks that GenAI performs are open-ended. | Provide detailed, multi-dimensional feedback that can be used to compare models, track improvements, and guide further development. | Evaluation is core to many of the other patterns and to building AI applications effectively. | [examples/17_llm_as_judge](examples/17_llm_as_judge) |
| 18 | Reflection | How to get the LLM to correct an earlier response in response to feedback or criticism. | The feedback is used to modify the prompt that is sent to the LLM a second time. | Reliable performance in most complex tasks where the approach can not be predetermined. | [examples/18_reflection](examples/18_reflection) |
| 19 | Dependency Injection | Need to independently develop and test each component of an LLM chain. | When you build chains of LLM calls, build them such that it is easy to inject a mock implementation to replace any step of the chain. | In any situation where you chain LLM calls or use external tools. | [examples/19_dependency_injection](examples/19_dependency_injection) |
| 20 | Prompt Optimization | Need to easily update prompts when dependencies change to maintain level of performance | Systematically set the prompts used in a GenAI pipeline by optimizing them on a dataset of examples | In any situation where you have to reduce the maintenance overhead associated with LLM version changes (and other dependencies). | [examples/20_prompt_optimiation](examples/20_prompt_optimization) |

</details>

<details>
<summary>Chapter 7: Enabling Agents to Take Action (Patterns 21-23) </summary>

| Pattern Number | Pattern Name | Problem | Solution | Usage Scenarios | Code Example |
| -------------: | :----------- | :------ | :------- | :-------------- | :----------- |  
| 21 | Tool Calling | How can you bridge the LLM and a software API so that the LLM is able to invoke the API and get the job done? | The LLM emits special tokens when it determines that a function needs to be called and also emits the parameters to pass to that function. A client-side postprocessor invokes the function with those parameters, and sends the results back to the LLM. The LLM incorporates the function results in its response. | Whenever you want the LLM to not just state the steps needed, but to execute those steps. Also allows you to incorporate up-to-date knowledge from real-time sources, connect to transactional enterprise systems, perform calculations, and use optimization solvers. | [examples/21_tool_calling](examples/21_tool_calling) |
| 22 | Code Execution | You have a software system that can do the task, but invoking it involves a DSL. | LLMs generate code that is then executed by an external system. | Creating graphs, annotating images, updating databases. | [examples/22_code_execution](examples/22_code_execution) |
| 23 | Multi-agent Collaboration | Handle multi-step tasks that require different tools, maintain content over extended interactions, evaluate situations and take appropriate actions without human intervention, and adapt to user preferences. | Multi-agent architectures allow you to solve real-world problems using specialized single-purpose agents and organizing them in ways that mimic human organizational structures. | Complex reasoning, multi-step problem solving, collaborative content creation, adversarial verification, specialized domain integration, self-improving systems | [examples/23_multi_agent](examples/23_multi_agent) |
    
</details>

<details>
<summary>Chapters 8: Addressing Constraints (Patterns 24-28) </summary>

| Pattern Number | Pattern Name               | Problem                                                                                                                   | Solution                                                                                                                                                                                                                                                                     | Usage Scenarios                                                                                                                           | Code Example                                                             |
| -------------: |:---------------------------|:--------------------------------------------------------------------------------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:------------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------|
| 24 | Small Language Model (SLM) | The foundational model you are using is introducing too much latency or cost.                                             | Use a small foundational model to fit within cost and latency constraints without compromising unduly on quality by employing quantization (reduce precision of model parameters), distillation (narrow knowledge scope), or speculative coding (backstop with larger model) | Narrow-scoped knowledge applications, cost reduction, edge device deployment, faster inference requirements, GPU-constrained environments | [examples/24_small_language_model](examples/24_small_language_model)     |
| 25 | Prompt Caching             | User requests follow patterns with repeated queries. Recomputing the same responses wastes resources and increases costs. | Reuse previously generated responses (in the case of client-side caching) and/or model internal states (in the case of server-side caching) for the same or similar prompts. The similarity can be based on prompt meaning (semantic cache) or overlap (prefix caching).     | Applications with repeated queries, cost optimization, interactive applications requiring fast responses, multi-tenant systems            | [examples/25_prompt_caching](examples/25_prompt_caching)                 |
| 26 | Inference Optimization     | Self-hosting LLMs brings with it GPU constraints and hardware utilization challenges. Real-time applications need faster response times. | Improves the efficiency of model inference by employing continuous batching (requests are pulled from a queue and slotted into GPU cores as soon as they become available), speculative decoding (efficiently compute the next set of tokens whenever the smaller model is able to do so, backstopping this with a large model), and/or prompt compression (preprocess prompts to make them shorter). | Self-hosted LLM deployments, real-time applications, GPU memory-constrained environments, high-throughput serving scenarios               | [examples/26_inference_optimization](examples/26_inference_optimization) |
| 27 | Degradation Testing        |  Need metrics to help identify when service quality degrades and the constraint under which the application is bounded. | A set of core metrics — Time-to-First-Token (TTFT), End-to-End Request Latency (EERL), Tokens per Second (TPS) — and a variety of scalability and resilience metrics can help identify degradation of service quality; targeted interventions can help improve specific metrics. | Pre-production testing, performance validation, bottleneck identification, capacity planning, ongoing monitoring and optimization.        | [examples/27_degradation_testing](examples/27_degradation_testing)       |
| 28 | Long-Term Memory | LLM applications need to simulate memory of past interactions by prepending relevant history to each prompt, but this approach can become costly and inefficient with long conversations due to context window limitations. | LLM applications use various types of memory – working, episodic, procedural, and semantic – to maintain context, recall past interactions, personalize responses, and retain key facts, respectively. | Chatbots, multi-step workflows, personalization, processing large documents | [examples/28_long_term_memory](examples/28_long_term_memory)             |
"""

In [61]:
target_patterns = json.load(open("../outputs/[25.10.23] - 03 - Optimized prompts/patterns/gen_ai_patterns_github_patterns.json"))

In [62]:
result = llm.predict(
    evaluation_prompt.format(
        correct_json=correct_patterns_3,
        generated_json=target_patterns
    )
)
print(result)

```json
{
  "precision": 0.875,
  "recall": 1.000,
  "f1": 0.933,
  "correct": [
    "Logits Masking",
    "Grammar",
    "Style Transfer",
    "Reverse Neutralization",
    "Content Optimization",
    "Basic RAG",
    "Semantic Indexing",
    "Indexing at Scale",
    "Index-aware Retrieval",
    "Node Postprocessing",
    "Trustworthy Generation",
    "Deep Search",
    "Chain of Thought (CoT)",
    "Tree of Thoughts (ToT)",
    "Adapter Tuning",
    "Evol-Instruct",
    "LLM-as-Judge",
    "Reflection",
    "Dependency Injection",
    "Prompt Optimization",
    "Tool Calling",
    "Code Execution",
    "Multi-agent Collaboration",
    "Small Language Model (SLM)",
    "Prompt Caching",
    "Inference Optimization",
    "Degradation Testing",
    "Long-Term Memory"
  ],
  "missing": [],
  "extra": [
    "Template Generation",
    "Assembled Reformat",
    "Self-Check",
    "Guardrails"
  ]
}
```


In [9]:
import os,json
def get_json_list_from_dir(dir_path):
    json_list = []
    for i, filename in enumerate(os.listdir(dir_path)):
        print(f'Processing file {i+1}: {filename}')
        if filename.endswith('.json'):
            file_path = os.path.join(dir_path, filename)
            with open(file_path, 'r') as f:
                data = json.load(f)
                json_list.extend(data)
    return json_list

json_list = get_json_list_from_dir("../outputs/[25.10.23] - 03 - Optimized prompts/patterns/")

Processing file 1: 3704435.pdf_patterns.json
Processing file 2: NeurIPS-2024-gorilla-large-language-model-connected-with-massive-apis-Paper-Conference.pdf_patterns.json
Processing file 3: NeurIPS-2020-retrieval-augmented-generation-for-knowledge-intensive-nlp-tasks-Paper.pdf_patterns.json
Processing file 4: 2112.09332v3.pdf_patterns.json
Processing file 5: 2402.02716v1.pdf_patterns.json
Processing file 6: AI Design Patterns_ Understanding RAG Pattern.pdf_patterns.json
Processing file 7: tacl_a_00605.pdf_patterns.json
Processing file 8: 1-s2.0-S0164121224003224-main.pdf_patterns.json
Processing file 9: IEEE_Software_19__ML_Patterns.pdf_patterns.json
Processing file 10: 2303.13173v1.pdf_patterns.json
Processing file 11: carta23a.pdf_patterns.json
Processing file 12: 2403.14403v2.pdf_patterns.json
Processing file 13: gen_ai_patterns_github_patterns.json
Processing file 14: 2403.10131v2.pdf_patterns.json
Processing file 15: Song_LLM-Planner_Few-Shot_Grounded_Planning_for_Embodied_Agents_wi

In [6]:
json_list

[{'Pattern Name': 'Tool-Augmented Foundation Model',
  'Problem': 'Foundation Models (FMs) have inherent limitations such as finite memorization capacity, potential for hallucination, lack of real-time or up-to-date knowledge, insufficient domain-specific expertise, limited interpretability, and susceptibility to adversarial attacks when operating in isolation.',
  'Context': 'AI systems are required to solve complex real-world tasks that demand capabilities beyond what a single Foundation Model can provide, necessitating interaction with external, specialized, and dynamic resources or precise execution of domain-specific operations.',
  'Solution': "Integrate specialized external tools with Foundation Models. The Foundation Model acts as a 'Controller' that understands user intent, plans, and invokes appropriate tools from a 'Tool Set' within an 'Environment.' A 'Perceiver' processes feedback from the environment and the user, summarizing it for the Controller.",
  'Result': 'Enhanced